In [7]:
spark.sparkContext.getConf().getOption("spark.serializer")

AttributeError: 'SparkConf' object has no attribute 'getOption'

In [17]:
print(spark.sparkContext._jsc.sc().listJars())

List(spark://192.168.111.100:60073/jars/org.apache.hadoop_hadoop-aws-3.3.4.jar, spark://192.168.111.100:60073/jars/com.amazonaws_aws-java-sdk-bundle-1.12.262.jar, spark://192.168.111.100:60073/jars/org.apache.iceberg_iceberg-spark-runtime-3.5_2.12-1.5.2.jar, spark://192.168.111.100:60073/jars/org.wildfly.openssl_wildfly-openssl-1.0.7.Final.jar)


In [18]:
spark._jvm.org.apache.hudi

In [1]:
spark._jvm.org.apache.hudi.common.util.HoodieTimer

NameError: name 'spark' is not defined

In [2]:
import os
import sys
from pyspark.sql import SparkSession

os.environ["PYSPARK_SUBMIT_ARGS"] = (
    '--conf spark.driver.extraClassPath="C:/data/spark/jars/iceberg-spark-runtime-4.0_2.13-1.10.0.jar" '
    "pyspark-shell"
)

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable


BUCKET_NAME = "rawload"
rawFiles = f"s3a://{BUCKET_NAME}"
WH_CATALOG_NAME = "raw"

spark = SparkSession.builder \
    .appName("Iceberg Reporting Access") \
    .config("spark.jars.packages", 
            "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.2,"
            "org.apache.hadoop:hadoop-aws:3.3.4,"
            "com.amazonaws:aws-java-sdk-bundle:1.12.262") \
    .config(f"spark.sql.catalog.{WH_CATALOG_NAME}", "org.apache.iceberg.spark.SparkCatalog") \
    .config(f"spark.sql.catalog.{WH_CATALOG_NAME}.catalog-impl", "org.apache.iceberg.hadoop.HadoopCatalog") \
    .config(f"spark.sql.catalog.{WH_CATALOG_NAME}.warehouse", rawFiles) \
    .config("spark.hadoop.fs.s3a.endpoint", "http://127.0.0.1:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .getOrCreate()

In [4]:
from pyspark.sql import SparkSession
import os
import sys

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

os.environ["PYSPARK_SUBMIT_ARGS"] = (
    '--jars "C:/data/spark/jars/hudi-spark4.0-bundle_2.13-1.2.0.jar" '
    '--conf spark.serializer=org.apache.spark.serializer.KryoSerializer '
    '--conf spark.kryo.registrator=org.apache.spark.HoodieSparkKryoRegistrar '
    'pyspark-shell'
)

BUCKET_NAME = "rawload"
rawFiles = f"s3a://{BUCKET_NAME}"

WH_CATALOG_NAME = "raw"

spark = (
    SparkSession.builder
    .appName("Iceberg Reporting Access")

    # Iceberg
    .config(
        f"spark.sql.catalog.{WH_CATALOG_NAME}",
        "org.apache.iceberg.spark.SparkCatalog"
    )
    .config(
        f"spark.sql.catalog.{WH_CATALOG_NAME}.catalog-impl",
        "org.apache.iceberg.hadoop.HadoopCatalog"
    )
    .config(
        f"spark.sql.catalog.{WH_CATALOG_NAME}.warehouse",
        rawFiles
    )

    # MinIO / S3A
    .config(
        "spark.hadoop.fs.s3a.endpoint",
        "http://127.0.0.1:9000"
    )
    .config(
        "spark.hadoop.fs.s3a.access.key",
        "minioadmin"
    )
    .config(
        "spark.hadoop.fs.s3a.secret.key",
        "minioadmin"
    )
    .config(
        "spark.hadoop.fs.s3a.path.style.access",
        "true"
    )
    .config(
        "spark.hadoop.fs.s3a.impl",
        "org.apache.hadoop.fs.s3a.S3AFileSystem"
    )
    .config(
        "spark.hadoop.fs.s3a.aws.credentials.provider",
        "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider"
    )

    # Iceberg
    .config(
        "spark.sql.extensions",
        "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions"
    )

    # Hudi
    .config(
    "spark.jars.packages",
    "org.apache.hudi:hudi-spark4.0-bundle_2.13:1.1.1"
    )

    .getOrCreate()
)

In [9]:
spark = (
    SparkSession.builder
    .appName("Hudi Test")

    .config(
        "spark.jars.packages",
        "org.apache.hudi:hudi-spark4.0-bundle_2.13:1.1.1"
    )

    # Hudi
    .config(
        "spark.sql.extensions",
        "org.apache.spark.sql.hudi.HoodieSparkSessionExtension"
    )
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.hudi.catalog.HoodieCatalog"
    )
    .config(
        "spark.serializer",
        "org.apache.spark.serializer.KryoSerializer"
    )
    .config(
        "spark.kryo.registrator",
        "org.apache.spark.HoodieSparkKryoRegistrar"
    )

    # MinIO
    .config(
        "spark.hadoop.fs.s3a.endpoint",
        "http://127.0.0.1:9000"
    )
    .config(
        "spark.hadoop.fs.s3a.access.key",
        "minioadmin"
    )
    .config(
        "spark.hadoop.fs.s3a.secret.key",
        "minioadmin"
    )
    .config(
        "spark.hadoop.fs.s3a.path.style.access",
        "true"
    )
    .config(
        "spark.hadoop.fs.s3a.impl",
        "org.apache.hadoop.fs.s3a.S3AFileSystem"
    )
    .config(
        "spark.hadoop.fs.s3a.aws.credentials.provider",
        "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider"
    )

    .getOrCreate()
)

In [5]:
BUCKET_NAME = "rawload"
rawFiles = f"s3a://{BUCKET_NAME}"
df = (
    spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(f"{rawFiles}/annual-enterprise-survey-2025-financial-year-provisional-size-bands.csv")
)

df.show()

+----+--------------------+--------------------+------------+--------------------+-----+-----------------+----+----+----+----+----+----+----+----+----+
|year|industry_code_ANZSIC|industry_name_ANZSIC|rme_size_grp|            variable|value|             unit| _c7| _c8| _c9|_c10|_c11|_c12|_c13|_c14|_c15|
+----+--------------------+--------------------+------------+--------------------+-----+-----------------+----+----+----+----+----+----+----+----+----+
|2011|                   A|Agriculture, Fore...|         a_0|       Activity unit|46134|            COUNT|NULL|NULL|NULL|NULL|NULL|NULL|NULL|NULL|NULL|
|2011|                   A|Agriculture, Fore...|         a_0|Rolling mean empl...|    0|            COUNT|NULL|NULL|NULL|NULL|NULL|NULL|NULL|NULL|NULL|
|2011|                   A|Agriculture, Fore...|         a_0|Salaries and wage...|  279|DOLLARS(millions)|NULL|NULL|NULL|NULL|NULL|NULL|NULL|NULL|NULL|
|2011|                   A|Agriculture, Fore...|         a_0|Sales, government...| 8187|

In [7]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Hudi MinIO Test")

    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.hudi.catalog.HoodieCatalog"
    )
    .config(
        "spark.sql.extensions",
        "org.apache.spark.sql.hudi.HoodieSparkSessionExtension"
    )

    # MinIO
    .config(
        "spark.hadoop.fs.s3a.endpoint",
        "http://127.0.0.1:9000"
    )
    .config(
        "spark.hadoop.fs.s3a.access.key",
        "minioadmin"
    )
    .config(
        "spark.hadoop.fs.s3a.secret.key",
        "minioadmin"
    )
    .config(
        "spark.hadoop.fs.s3a.path.style.access",
        "true"
    )
    .config(
        "spark.hadoop.fs.s3a.impl",
        "org.apache.hadoop.fs.s3a.S3AFileSystem"
    )
    .config(
        "spark.hadoop.fs.s3a.aws.credentials.provider",
        "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider"
    )
    .getOrCreate()
)

In [11]:
spark.sparkContext.getConf().get(
    "spark.serializer",
    "NOT SET"
)

'NOT SET'

In [12]:
spark.sparkContext.getConf().get(
    "spark.kryo.registrator",
    "NOT SET"
)

'NOT SET'

In [13]:
print("Serializer:",
      spark.sparkContext.getConf().get(
          "spark.serializer", "NOT SET"
      ))

print("Kryo Registrator:",
      spark.sparkContext.getConf().get(
          "spark.kryo.registrator", "NOT SET"
      ))

Serializer: NOT SET
Kryo Registrator: NOT SET


In [17]:
import os

print("PYSPARK_SUBMIT_ARGS =")
print(os.environ.get("PYSPARK_SUBMIT_ARGS"))

print("\nSPARK_HOME =")
print(os.environ.get("SPARK_HOME"))

print("\nPYSPARK_PYTHON =")
print(os.environ.get("PYSPARK_PYTHON"))

print("\nPYSPARK_DRIVER_PYTHON =")
print(os.environ.get("PYSPARK_DRIVER_PYTHON"))


PYSPARK_SUBMIT_ARGS =
--jars "C:/data/spark/jars/hudi-spark4.0-bundle_2.13-1.2.0.jar" --conf spark.serializer=org.apache.spark.serializer.KryoSerializer --conf spark.kryo.registrator=org.apache.spark.HoodieSparkKryoRegistrar pyspark-shell

SPARK_HOME =
C:\data\spark

PYSPARK_PYTHON =
c:\data\python\python.exe

PYSPARK_DRIVER_PYTHON =
c:\data\python\python.exe


In [18]:
print(os.environ.get("PYSPARK_SUBMIT_ARGS", "").split())

['--jars', '"C:/data/spark/jars/hudi-spark4.0-bundle_2.13-1.2.0.jar"', '--conf', 'spark.serializer=org.apache.spark.serializer.KryoSerializer', '--conf', 'spark.kryo.registrator=org.apache.spark.HoodieSparkKryoRegistrar', 'pyspark-shell']


In [14]:
print(spark.sparkContext._jsc.sc().listJars())

List(spark://192.168.111.100:61197/jars/org.wildfly.openssl_wildfly-openssl-1.0.7.Final.jar, spark://192.168.111.100:61197/jars/org.apache.hadoop_hadoop-aws-3.3.4.jar, spark://192.168.111.100:61197/jars/com.amazonaws_aws-java-sdk-bundle-1.12.262.jar, spark://192.168.111.100:61197/jars/org.apache.iceberg_iceberg-spark-runtime-3.5_2.12-1.5.2.jar)


In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Hudi MinIO Test")
    .config(
        "spark.sql.extensions",
        "org.apache.spark.sql.hudi.HoodieSparkSessionExtension"
    )
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.hudi.catalog.HoodieCatalog"
    )
    .getOrCreate()
)

PySparkRuntimeError: [JAVA_GATEWAY_EXITED] Java gateway process exited before sending its port number.

In [1]:
import os

os.environ.pop("PYSPARK_SUBMIT_ARGS", None)

'--jars C:/data/spark/jars/iceberg-spark-runtime-4.0_2.13-1.10.0.jar pyspark-shell'

In [2]:
import sys
from pyspark.sql import SparkSession

HUDI_JAR = r"C:\data\spark\jars\hudi-spark4.0-bundle_2.13-1.2.0.jar"

spark = (
    SparkSession.builder
    .appName("Hudi MinIO Test")
    .config("spark.jars", HUDI_JAR)
    .config(
        "spark.serializer",
        "org.apache.spark.serializer.KryoSerializer"
    )
    .config(
        "spark.kryo.registrator",
        "org.apache.spark.HoodieSparkKryoRegistrar"
    )
    .config(
        "spark.sql.extensions",
        "org.apache.spark.sql.hudi.HoodieSparkSessionExtension"
    )
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.hudi.catalog.HoodieCatalog"
    )

    # MinIO
    .config(
        "spark.hadoop.fs.s3a.endpoint",
        "http://127.0.0.1:9000"
    )
    .config(
        "spark.hadoop.fs.s3a.access.key",
        "minioadmin"
    )
    .config(
        "spark.hadoop.fs.s3a.secret.key",
        "minioadmin"
    )
    .config(
        "spark.hadoop.fs.s3a.path.style.access",
        "true"
    )
    .config(
        "spark.hadoop.fs.s3a.impl",
        "org.apache.hadoop.fs.s3a.S3AFileSystem"
    )
    .config(
        "spark.hadoop.fs.s3a.aws.credentials.provider",
        "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider"
    )
    .getOrCreate()
)

In [3]:
print(spark.version)

print(
    spark.sparkContext.getConf().get(
        "spark.serializer",
        "NOT SET"
    )
)

print(spark.sparkContext._jsc.sc().listJars())

4.0.1
org.apache.spark.serializer.KryoSerializer
List(spark://192.168.111.100:50898/jars/hudi-spark4.0-bundle_2.13-1.2.0.jar)


In [7]:
from pyspark.sql import functions as F

df_bronze = df.withColumn(
    "_ingestion_id",
    F.expr("uuid()")
)

In [8]:
hudi_path = "s3a://bronzeload/enterprise_survey"

(
    df_bronze.write
    .format("org.apache.hudi")
    .option("hoodie.table.name", "enterprise_survey")
    .option(
        "hoodie.datasource.write.table.type",
        "COPY_ON_WRITE"
    )
    .option(
        "hoodie.datasource.write.recordkey.field",
        "_ingestion_id"
    )
    .option(
        "hoodie.datasource.write.precombine.field",
        "_ingestion_id"
    )
    .mode("overwrite")
    .save(hudi_path)
)

In [9]:
spark.read.format("org.apache.hudi").load(hudi_path).show()

+-------------------+--------------------+--------------------+----------------------+--------------------+----+--------------------+--------------------+------------+--------------------+-----+-----------------+----+----+----+----+----+----+----+----+----+--------------------+
|_hoodie_commit_time|_hoodie_commit_seqno|  _hoodie_record_key|_hoodie_partition_path|   _hoodie_file_name|year|industry_code_ANZSIC|industry_name_ANZSIC|rme_size_grp|            variable|value|             unit| _c7| _c8| _c9|_c10|_c11|_c12|_c13|_c14|_c15|       _ingestion_id|
+-------------------+--------------------+--------------------+----------------------+--------------------+----+--------------------+--------------------+------------+--------------------+-----+-----------------+----+----+----+----+----+----+----+----+----+--------------------+
|  20260815133731441|20260815133731441...|21670bd2-b44c-4e3...|                      |c2e633f1-37f6-44d...|2017|                   N|Administrative an...|      h_2